# NB03 (V3_1) — Frozen warm-start M2: full-mosaic Musanga probabilities

**Selected production candidate:** CV-fold-0 warm-start, fine-tuned two epochs on all 30 circles, versioned epoch-02 checkpoint. Runs **native 256×256 RGB** tiles, **128px overlap**, **the same positive feather weights used in NB02B Sections 15/19/20**, and outputs georeferenced uint16 probabilities at scale 10,000 with NoData 65,535.

**Run order:** Section 00 (user settings), Sections 01–03 (setup + seam preflight), 04 (restartable full mosaics), 05 (final QA). **Default: Baego only**; change `RUN_ALL_SITES=True` in Section 00 after verifying its map and circle QA. MOS includes both MOS_1_2 and MOS_5_6 and optionally MOS_3_4, which was excluded from reference training. Historical outputs are preserved. Prior NB03 archived at `V3_1/archive/03_Full_Mosaic_M2_Inference_v3_1_pre_warmstart.ipynb`.

**Important:** NB02B spatial-CV provides independent validation. Warm-start all-30 circle checks are in-sample; a successful NB03 run confirms implementation consistency, not independent generalization.



## 00 — RUN SETTINGS (edit here before running)
All user-facing switches live here. **Default: Baego only** for the first technical smoke test. Set `RUN_ALL_SITES=True` after Baego QA passes.


In [ ]:
# ===== USER SETTINGS =====
RUN_ALL_SITES = False  # False: Baego only | True: all audited mosaics
QA_CIRCLE_ID = 0       # Baego exhaustive reference circle (original row)
QA_WINDOW_PIXELS = 1024  # display/score an image crop centered on this circle
QA_CIRCLES_PER_SITE = 2  # Section 07: sample per site (max 7 Baego, 11 MOS, 7 Yangambi, 5 Yoko)
QA_RANDOM_SEED = 20261021  # reproducible selection
assert isinstance(RUN_ALL_SITES, bool)
print('RUN_ALL_SITES =', RUN_ALL_SITES)


## 01 — Input audit and frozen-checkpoint verification


In [ ]:
%pip -q install rasterio segmentation-models-pytorch pandas matplotlib
import json,hashlib,os,shutil,time,math,warnings,gc
from pathlib import Path
import numpy as np,pandas as pd,rasterio,torch
from rasterio.windows import Window
from rasterio.errors import NodataShadowWarning
from tqdm.auto import tqdm
from IPython.display import display
from google.colab import drive
warnings.filterwarnings("once",category=NodataShadowWarning)
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
ROOT=next((p for p in (Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final"),Path("/content/drive/MyDrive/Musanga/Final")) if p.is_dir()),None)
assert ROOT is not None,"Missing Musanga/Final on Drive"
AUDIT=ROOT/"Outputs/V3_1/00_input_audit/00_input_manifest.json"
BASE=ROOT/"Outputs/V3_1/02B_spatial3fold_no_MOS34"
assert AUDIT.exists() and BASE.exists(),("Missing audited V3_1 inputs",AUDIT,BASE)
A=json.loads(AUDIT.read_text())
assert A.get("schema_version")=="v3_1_input_audit_kvo_final"
assert A.get("reference_version")=="KVO_field_final"
assert torch.cuda.is_available(),"Choose a Colab GPU runtime"
import segmentation_models_pytorch as smp
class ResNet18UNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
    def forward(self,x):return self.net(x)[:,0]
MOSAICS={m["file"]:Path(m["path"]) for m in A["mosaics"]}
EXPECTED={"Baego.tif","Mos_1_2_transparent_mosaic_group1_georef.tif",
          "Mos_3_4_transparent_mosaic_group1_georef.tif",
          "Mos_5_6_transparent_mosaic_group1_georef.tif",
          "Yangambi_georef.tif","Yoko_kvo_georef.tif"}
assert EXPECTED.issubset(MOSAICS),("Missing full mosaics",sorted(EXPECTED-set(MOSAICS)))
assert all(p.exists() for p in MOSAICS.values()),"Missing mosaic on Drive"
RUN_MOSAICS=list(MOSAICS)
def sha_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for piece in iter(lambda:f.read(8*1024*1024),b""):h.update(piece)
    return h.hexdigest()
def shapefile_content_signature(source):
    p=Path(source)
    assert p.exists(),p
    files=sorted(p.parent.glob(p.stem+".*")) if p.suffix.lower()==".shp" else [p]
    files=([x for x in files if x.suffix.lower() in (".shp",".shx",".dbf",".prj",".cpg")]
           if p.suffix.lower()==".shp" else [p])
    h=hashlib.sha256()
    for f in files:
        h.update(f.suffix.lower().encode())
        with f.open("rb") as stream:
            for chunk in iter(lambda:stream.read(4*1024*1024),b""):h.update(chunk)
    return h.hexdigest()
# Frozen selected model, verified against the provenance and QA created in NB02B.
FINAL_DIR=BASE/"final_warmstart_M2_c6a33504c37efb2ebb84"
MANIFEST=FINAL_DIR/"manifest.json"
CHECKPOINT=FINAL_DIR/"M2_warmstart_all30_epoch02.pt"
START_CV=BASE/"fold_0/M2_resnet18_spatialCV_3ee0e5ba7eea84591d7b.pt"
QA_DIR=FINAL_DIR/"QA"
assert all(p.is_file() for p in (MANIFEST,CHECKPOINT,START_CV,
    QA_DIR/"19_baego_warmstart_by_circle.csv",
    QA_DIR/"20_other_sites_warmstart_by_circle.csv"))
config=json.loads(MANIFEST.read_text())
assert config["version"]=="M2_V3_1_warmstart_cv0_to_all30_v1"
assert config["source_cv"]==str(START_CV)
assert config["crop"]==256 and config["batch"]==8 and config["epochs"]==2
assert abs(config["lr"]-2e-5)<1e-12
assert config["label_sha"]==shapefile_content_signature(
    A["exhaustive_annotations"]["positive_polygons"]), "Annotations changed; STOP"
TRAINING_KEY=FINAL_DIR.name.removeprefix("final_warmstart_M2_")
MODEL_MODE="warmstart_epoch02"
CHECKPOINTS=[dict(fold="WARMSTART_EPOCH02",path=str(CHECKPOINT),sha256=sha_file(CHECKPOINT))]
ref=dict(sha256=config["label_sha"],circles=30)
print("Frozen warm-start epoch-02 model verified:",TRAINING_KEY,flush=True)
print("Checkpoint SHA256:",CHECKPOINTS[0]["sha256"],flush=True)
# Baego is a technical smoke test. Switch to True only after inspecting output.
assert "RUN_ALL_SITES" in globals(), "Run Section 00 — USER SETTINGS first"
RUN_MOSAICS=(list(MOSAICS) if RUN_ALL_SITES else ["Baego.tif"])
print("Mosaics scheduled:",RUN_MOSAICS)
TILE=256;STEP=128;CORE=512;STRIPE=1024;BATCH=12
SCALE=10000;NODATA=65535;EDGE_FLOOR=0.05
PIPELINE="NB03_V3_1_M2_warmstart_"+TRAINING_KEY+"_native256_stride128_feather_f32_v1"
assert STEP==TILE//2 and CORE%STEP==0 and STRIPE%CORE==0
OUT=ROOT/("Outputs/V3_1/03_M2_warmstart_"+TRAINING_KEY+"_full_mosaic")
MAPS=OUT/"probability_maps";PARTS=OUT/"stripes"
MAPS.mkdir(parents=True,exist_ok=True);PARTS.mkdir(parents=True,exist_ok=True)
SCRATCH=Path("/content/nb03_scratch");SCRATCH.mkdir(exist_ok=True)
ENSEMBLE_SHA=hashlib.sha256("".join(z["sha256"] for z in CHECKPOINTS).encode()).hexdigest()


## 02 — Native-resolution 256-pixel inference with globally aligned cosine blending
Every mosaic pixel receives contributions from the same global tile lattice regardless of output block or stripe position. Checkpoint probabilities are averaged when multiple checkpoints are selected, never their logits. This is computationally expensive but avoids spatial-window mismatch from V2.


In [ ]:
MODELS=[]
for x in CHECKPOINTS:
    model=ResNet18UNet().cuda()
    model.load_state_dict(torch.load(x["path"],map_location="cpu",weights_only=True),strict=True)
    model.eval()
    MODELS.append(model)
assert len(MODELS)==len(CHECKPOINTS) and len(MODELS) in (1,3)
# EXACT NB02B tiled-256 QA feather, including its positive edge floor.
u=np.arange(TILE,dtype="float32")
u=np.minimum(1.,np.maximum(0.05,np.minimum((u+.5)/32.,(TILE-u-.5)/32.)))
WEIGHT=np.outer(u,u).astype("float32")
assert WEIGHT.min()>0
def valid_rgb(raw):
    # MATCH NB02B: masked RGB bands, excluding exact all-black, all-white, blue.
    a=raw.filled(0).astype("float32")
    v=~np.any(np.ma.getmaskarray(raw),axis=0)
    v &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    v &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return v

def starts(lo,hi):
    return list(range((lo-TILE+1)//STEP*STEP,(hi-1)//STEP*STEP+1,STEP))

def infer_rect(src,r0,c0,h,w):
    """Globally anchored overlapping 256-pixel windows; no core-dependent seams."""
    assert 0<=r0<src.height and 0<=c0<src.width
    assert 0<h<=src.height-r0 and 0<w<=src.width-c0
    center=src.read([1,2,3],window=Window(c0,r0,w,h),masked=True)
    valid=valid_rgb(center)
    if not valid.any():
        return np.full((h,w),np.nan,"float32"),valid
    rs=starts(r0,r0+h);cs=starts(c0,c0+w)
    top=min(rs);left=min(cs);bottom=max(rs)+TILE;right=max(cs)+TILE
    ext=np.zeros((3,bottom-top,right-left),dtype="uint8")
    rr0=max(0,top);cc0=max(0,left)
    rr1=min(bottom,src.height);cc1=min(right,src.width)
    if rr1>rr0 and cc1>cc0:
        src_data=src.read([1,2,3],window=Window(cc0,rr0,cc1-cc0,rr1-rr0),masked=True)
        ext[:,rr0-top:rr1-top,cc0-left:cc1-left]=src_data.filled(0)
    total=np.zeros((h,w),dtype="float64")
    weights_total=np.zeros((h,w),dtype="float64")
    patches=[];tasks=[]
    def flush():
        if not patches:return
        batch=torch.from_numpy(np.stack(patches).astype("float32")/255.).cuda()
        ensemble=None
        with torch.inference_mode():
            for model in MODELS:
                # Exact NB02B evaluation: FP32; no autocast.
                prediction=torch.sigmoid(model(batch)).float()
                ensemble=prediction if ensemble is None else ensemble+prediction
            predictions=(ensemble/len(MODELS)).cpu().numpy()
        for (tr,tc,cr0,cr1,ccr0,ccr1),prob in zip(tasks,predictions):
            pr0=cr0-(tr-r0);pc0=ccr0-(tc-c0)
            hh=cr1-cr0;ww=ccr1-ccr0
            wwgt=WEIGHT[pr0:pr0+hh,pc0:pc0+ww]
            pp=prob[pr0:pr0+hh,pc0:pc0+ww]
            total[cr0:cr1,ccr0:ccr1]+=wwgt*pp
            weights_total[cr0:cr1,ccr0:ccr1]+=wwgt
        patches.clear();tasks.clear()
    for tr in rs:
        for tc in cs:
            cr0=max(0,tr-r0);cr1=min(h,tr+TILE-r0)
            ccr0=max(0,tc-c0);ccr1=min(w,tc+TILE-c0)
            if cr0>=cr1 or ccr0>=ccr1:continue
            if not valid[cr0:cr1,ccr0:ccr1].any():continue
            patch=ext[:,tr-top:tr-top+TILE,tc-left:tc-left+TILE]
            assert patch.shape==(3,TILE,TILE)
            patches.append(patch);tasks.append((tr,tc,cr0,cr1,ccr0,ccr1))
            if len(patches)>=BATCH:flush()
    flush()
    assert np.all(weights_total[valid]>0),("Missing tile weight",r0,c0)
    p=np.full((h,w),np.nan,"float32")
    p[valid]=(total[valid]/weights_total[valid]).astype("float32")
    assert np.all(np.isfinite(p[valid])) and np.all((p[valid]>=0)&(p[valid]<=1))
    return p,valid

def to_u16(prob,valid):
    arr=np.full(prob.shape,NODATA,dtype="uint16")
    arr[valid]=np.rint(prob[valid]*SCALE).astype("uint16")
    assert np.array_equal(arr!=NODATA,valid)
    return arr


## 03 — Mandatory preflight against tiling seams
For each source mosaic, compute the same pixels in two **different, overlapping 512×512 output blocks**. The predicted values must match to 1e−4 despite shifted chunk boundaries. This catches incorrect tile-grid resets before any full-mosaic computation.


In [ ]:
# Required seam preflight. Find representative VALID areas by scanning the
# mosaic on a coarse grid, not by guessing four diagonal locations.
# Check comparison geometry on real valid pixels; never silently skip a site.
PREFLIGHT_OK=False
PREFLIGHT=[]
for name in RUN_MOSAICS:
    with rasterio.open(MOSAICS[name]) as ds:
        assert ds.count>=3 and all(x=="uint8" for x in ds.dtypes[:3]),(name,ds.dtypes)
        assert ds.crs is not None
        if ds.height<512 or ds.width<768:
            raise RuntimeError(("Source too small for this 512px/768px seam diagnostic",name,ds.shape))

        # Survey 9x9 candidate origins across the true source extent.
        # Each probe reads only 256x256 pixels in the actual overlap, making
        # it much cheaper than running the neural network for all 81 locations.
        candidates=[]
        rows=np.unique(np.linspace(0,ds.height-512,9).astype(int))
        cols=np.unique(np.linspace(0,ds.width-768,9).astype(int))
        for row in rows:
            for col in cols:
                center=ds.read([1,2,3],window=Window(int(col)+256,int(row),256,256),masked=True)
                count=int(valid_rgb(center).sum())
                if count:
                    candidates.append((count,int(row),int(col)))
        candidates.sort(reverse=True)
        if not candidates:
            # No arbitrary lowering of the threshold: inspect the actual RGB
            # mask and source metadata if this case ever occurs.
            raise RuntimeError((
                "No valid RGB pixels in any of 81 spread-out overlap probes",
                name,ds.width,ds.height,ds.nodata,ds.mask_flag_enums))
        best_count,row,col=candidates[0]
        print(f"PREFLIGHT {name}: best 256x256 overlap probe = "
              f"{best_count:,}/65,536 valid pixels; row={row}, col={col}",flush=True)

        # Different output-core origins, identical global input tile lattice.
        # Pixels in the shared 512x256 area MUST have identical probabilities.
        p0,v0=infer_rect(ds,row,col,512,512)
        p1,v1=infer_rect(ds,row,col+256,512,512)
        both=v0[:,256:]&v1[:,:256]
        n=int(both.sum())
        assert n>=best_count,(name,"unexpected mask difference",n,best_count)
        difference=np.abs(p0[:,256:][both]-p1[:,:256][both])
        maxdiff=float(difference.max())
        meandiff=float(difference.mean())
        assert np.isfinite(maxdiff) and maxdiff<1e-4,(
            "SEAM FAILURE: identical geographic pixels differ across output-block origins",
            name,maxdiff,meandiff,n)
        PREFLIGHT.append(dict(mosaic=name,crs=str(ds.crs),valid_probe_pixels=best_count,
               common_valid_pixels=n,max_abs_probability_diff=maxdiff,
               mean_abs_probability_diff=meandiff,row=row,col=col,status="PASS"))
        print(f"PASS {name}: {n:,} common valid pixels; "
              f"max difference {maxdiff:.8f}",flush=True)

assert len(PREFLIGHT)==len(RUN_MOSAICS)
display(pd.DataFrame(PREFLIGHT))
(OUT/"03_seam_preflight.json").write_text(json.dumps(PREFLIGHT,indent=2))
PREFLIGHT_OK=True
print("All mosaic seam tests PASS; production is now enabled.")


## 04 — Full mosaics: atomic restartable stripes and protected final GeoTIFFs
Every 512×512 core is written, including fully invalid cores (NoData=65535). Completed 1024-row stripes persist on Google Drive. Re-execution reuses compatible stripes instead of recomputing neural inference. An incompatible existing map or stripe triggers a stop rather than overwrite.


In [ ]:
assert globals().get("PREFLIGHT_OK",False),"Run seam preflight before production"
def out_profile(ds,height=None,transform=None):
    out=ds.profile.copy()
    for k in ("photometric","alpha","extrasamples","NBITS","colorinterp","interleave"):
        out.pop(k,None)
    out.update(driver="GTiff",dtype="uint16",count=1,nodata=NODATA,
               tiled=True,blockxsize=256,blockysize=256,compress="deflate",
               predictor=2,bigtiff="YES",interleave="band",SPARSE_OK=False)
    if height is not None:out["height"]=int(height)
    if transform is not None:out["transform"]=transform
    return out

def spec_for(name):
    p=MOSAICS[name];st=p.stat()
    with rasterio.open(p) as ds:
        spec=dict(name=name,path=str(p),bytes=st.st_size,mtime_ns=st.st_mtime_ns,
             width=ds.width,height=ds.height,crs=str(ds.crs),
             transform=list(ds.transform)[:6],ensemble_sha256=ENSEMBLE_SHA,training_key=TRAINING_KEY,
             annotation_sha256=ref["sha256"],input_manifest_sha256=sha_file(AUDIT),
             training_manifest_sha256=sha_file(MANIFEST),pipeline=PIPELINE,
             tile=TILE,step=STEP,core=CORE,stripe=STRIPE,
             scale=SCALE,nodata=NODATA,edge_floor=EDGE_FLOOR)
    spec["key"]=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:24]
    return spec

def same_transform(a,b):
    return np.allclose(list(a)[:6],list(b)[:6],atol=1e-7,rtol=0)
def stripe_ok(path,spec,row,h):
    if not path.exists():return False
    try:
        with rasterio.open(path) as ds:
            target=rasterio.windows.transform(
                Window(0,row,spec["width"],h),rasterio.Affine(*spec["transform"]))
            return (ds.count==1 and ds.dtypes[0]=="uint16" and ds.nodata==NODATA
               and ds.width==spec["width"] and ds.height==h
               and str(ds.crs)==spec["crs"] and same_transform(ds.transform,target)
               and ds.tags().get("SPEC")==spec["key"]
               and ds.tags().get("ROW")==str(row) and ds.tags().get("COMPLETE")=="YES")
    except Exception:return False
def full_ok(path,spec):
    if not path.exists():return False
    try:
        with rasterio.open(path) as ds:
            return (ds.count==1 and ds.dtypes[0]=="uint16" and ds.nodata==NODATA
               and ds.width==spec["width"] and ds.height==spec["height"]
               and str(ds.crs)==spec["crs"]
               and same_transform(ds.transform,rasterio.Affine(*spec["transform"]))
               and ds.tags().get("SPEC")==spec["key"] and ds.tags().get("COMPLETE")=="YES")
    except Exception:return False

def run_one(name):
    spec=spec_for(name)
    target=MAPS/(Path(name).stem+"_M2_" + MODEL_MODE + "_probability_u16_FULL.tif")
    if target.exists():
        assert full_ok(target,spec),("Refusing to overwrite incompatible map",target)
        print("VALID EXISTING MAP:",target,flush=True)
        return dict(mosaic=name,status="REUSED",file=str(target))
    folder=PARTS/Path(name).stem/spec["key"]
    folder.mkdir(parents=True,exist_ok=True)
    config=folder/"spec.json"
    if config.exists():assert json.loads(config.read_text())==spec
    else:config.write_text(json.dumps(spec,indent=2))
    nstripes=math.ceil(spec["height"]/STRIPE);created=0;reused=0
    started=time.perf_counter()
    with rasterio.open(MOSAICS[name]) as src:
        for ii,row in enumerate(range(0,src.height,STRIPE)):
            height=min(STRIPE,src.height-row)
            dest=folder/f"stripe_{ii:05d}.tif"
            if stripe_ok(dest,spec,row,height):
                reused+=1
                print(name,f"stripe {ii+1}/{nstripes} CACHE HIT",flush=True)
                continue
            assert not dest.exists(),("Incompatible stripe; inspect rather than delete",dest)
            local=SCRATCH/f"{Path(name).stem}_{ii:05d}.tif"
            local.unlink(missing_ok=True)
            count=0;blocks=0
            try:
                trans=src.window_transform(Window(0,row,src.width,height))
                with rasterio.open(local,"w",**out_profile(src,height,trans)) as dst:
                    for rr in range(row,row+height,CORE):
                        hh=min(CORE,row+height-rr)
                        for cc in range(0,src.width,CORE):
                            ww=min(CORE,src.width-cc)
                            prob,valid=infer_rect(src,rr,cc,hh,ww)
                            dst.write(to_u16(prob,valid),1,window=Window(cc,rr-row,ww,hh))
                            count+=int(valid.sum());blocks+=1
                    dst.update_tags(SPEC=spec["key"],ROW=str(row),
                                    COMPLETE="YES",VALID_PIXELS=str(count),
                                    MODEL_SHA256=ENSEMBLE_SHA,TRAINING_KEY=TRAINING_KEY,
                                    ANNOTATION_SHA256=ref["sha256"])
                assert blocks==math.ceil(height/CORE)*math.ceil(src.width/CORE)
                assert stripe_ok(local,spec,row,height),("Local stripe failed QA",local)
                pending=dest.with_name(dest.name+".uploading")
                pending.unlink(missing_ok=True)
                shutil.copyfile(local,pending)
                os.replace(pending,dest)
                assert stripe_ok(dest,spec,row,height),("Stripe upload failed",dest)
                created+=1
                print(name,f"stripe {ii+1}/{nstripes} READY; {count:,} valid pixels",flush=True)
            finally:
                local.unlink(missing_ok=True)
    assert all(stripe_ok(folder/f"stripe_{ii:05d}.tif",spec,row,
               min(STRIPE,spec["height"]-row))
               for ii,row in enumerate(range(0,spec["height"],STRIPE)))
    assembly=MAPS/(Path(name).stem+"_M2_" + MODEL_MODE + "_probability_u16_FULL.assembling.tif")
    assembly.unlink(missing_ok=True)
    with rasterio.open(MOSAICS[name]) as src,rasterio.open(assembly,"w",**out_profile(src)) as out:
        for ii,row in enumerate(tqdm(range(0,src.height,STRIPE),desc=name+" assembly")):
            with rasterio.open(folder/f"stripe_{ii:05d}.tif") as stripe:
                for r in range(0,stripe.height,256):
                    hh=min(256,stripe.height-r)
                    a=stripe.read(1,window=Window(0,r,stripe.width,hh))
                    out.write(a,1,window=Window(0,row+r,stripe.width,hh))
        out.update_tags(SPEC=spec["key"],COMPLETE="YES",
                        COVERAGE="ALL_NATIVE_VALID_RGB",ENSEMBLE_SHA256=ENSEMBLE_SHA,
                        PROBABILITY_SCALE=str(SCALE),TRAINING_KEY=TRAINING_KEY,
                        ANNOTATION_SHA256=ref["sha256"],CHECKPOINT_SHA256=CHECKPOINTS[0]["sha256"])
    # Internal pyramid overviews: fast QGIS/ArcGIS rendering, no loss of full resolution.
    from rasterio.enums import Resampling
    with rasterio.open(assembly,"r+") as pyramid:
        factors=[f for f in (2,4,8,16,32,64,128) if min(pyramid.width,pyramid.height)//f>=128]
        if factors:
            pyramid.build_overviews(factors,Resampling.average)
            pyramid.update_tags(ns="rio_overview",resampling="average")
    with rasterio.open(assembly) as pyramid_check:
        assert pyramid_check.overviews(1)==factors
    assert full_ok(assembly,spec),("Assembled map failed QA",assembly)
    assert not target.exists(),"Refusing to overwrite final product"
    os.replace(assembly,target)
    assert full_ok(target,spec)
    result=dict(mosaic=name,status="COMPLETE",file=str(target),
        crs=spec["crs"],width=spec["width"],height=spec["height"],
        stripes=nstripes,new_stripes=created,reused_stripes=reused,
        runtime_min=round((time.perf_counter()-started)/60,2))
    (OUT/(Path(name).stem+"_QA.json")).write_text(json.dumps(result,indent=2))
    return result

RUN_RESULTS=[]
for name in RUN_MOSAICS:
    RUN_RESULTS.append(run_one(name))
display(pd.DataFrame(RUN_RESULTS))
(OUT/"03_run_manifest.json").write_text(json.dumps(dict(
    checkpoints=CHECKPOINTS,run_results=RUN_RESULTS,
    ensemble_sha256=ENSEMBLE_SHA,pipeline=PIPELINE,
    reference_sha256=ref["sha256"],training_key=TRAINING_KEY,
    training_manifest=str(MANIFEST),input_manifest=str(AUDIT)),indent=2))


## 05 — Final full-map QA and visual overviews
Compare native CRS, affine transform and grid dimensions; verify source-versus-output valid-pixel masks at the image edges and either side of every stripe boundary. Save small probability overviews to Drive. Inspect actual RGB/probability overlays in GIS before declaring scientific production finished.


In [ ]:
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
QA=[]
for name in RUN_MOSAICS:
    spec=spec_for(name)
    output=MAPS/(Path(name).stem+"_M2_" + MODEL_MODE + "_probability_u16_FULL.tif")
    assert full_ok(output,spec),("Incomplete/missing output",name)
    with rasterio.open(MOSAICS[name]) as original,rasterio.open(output) as pred:
        assert pred.overviews(1),("Missing internal GeoTIFF pyramids",name)
        # Source and output must match EXACTLY in native raster grid.
        assert pred.width==original.width and pred.height==original.height
        assert pred.crs==original.crs and pred.transform==original.transform
        # Validate validity at distributed output blocks, including stripe seams.
        points=[(0,0),(original.height-1,original.width-1),
                (original.height//2,original.width//2)]
        for seam in range(STRIPE,original.height,STRIPE):
            for delta in (-1,0,1):
                if 0<=seam+delta<original.height:
                    points.append((seam+delta,original.width//2))
        tested=0;good=0
        for rr,cc in points:
            h=min(256,original.height-rr)
            w=min(256,original.width-cc)
            win=Window(cc,rr,w,h)
            rgb=original.read([1,2,3],window=win,masked=True)
            mask=valid_rgb(rgb)
            z=pred.read(1,window=win)
            assert np.array_equal(z!=NODATA,mask),(
                "Full-map validity mask differs from NB02B-style RGB validity",name,rr,cc)
            assert np.all(z[mask]<=SCALE)
            tested+=1;good+=int(mask.sum())
        factor=max(1,math.ceil(max(pred.width,pred.height)/900))
        overview=pred.read(1,out_shape=(max(1,pred.height//factor),
                     max(1,pred.width//factor)),resampling=Resampling.nearest)
        shown=np.where(overview==NODATA,np.nan,overview.astype("float32")/SCALE)
        fig,ax=plt.subplots(figsize=(7,5))
        im=ax.imshow(shown,cmap="viridis",vmin=0,vmax=1)
        fig.colorbar(im,ax=ax,label="Musanga probability")
        ax.set_title(Path(name).stem+" — M2 " + MODEL_MODE)
        ax.set_axis_off()
        fig.savefig(OUT/(Path(name).stem+"_overview.png"),dpi=140,bbox_inches="tight")
        plt.show();plt.close(fig)
        QA.append(dict(mosaic=name,crs=str(pred.crs),width=pred.width,
            height=pred.height,file_GB=round(output.stat().st_size/1e9,2),
            tested_windows=tested,tested_valid_pixels=good,
            overview_valid_fraction=float(np.mean(overview!=NODATA))))
display(pd.DataFrame(QA))
(OUT/"03_final_QA.json").write_text(json.dumps(QA,indent=2))
print("Before ecological analysis: inspect seam lines and MOS3–4 qualitatively in QGIS/ArcGIS.")


## 06 — Quick visual & quantitative QA on a Baego reference circle
Read the **finished GeoTIFF** directly. RGB / Musanga annotation / predicted probability side by side; calculate window-intersection cover bias, pixel precision, recall, F1 and IoU. The annotation mask is restricted to the selected exhaustive circle and ROI. **Diagnostic, in-sample**, not an independent accuracy measure. Change circle and window size in Section 00. No neural-network inference.


In [ ]:
# SECTION 06 — Finished GeoTIFF vs annotated Baego circle (READ ONLY)
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.windows import Window
import matplotlib.pyplot as plt
from shapely.geometry import box
assert "QA_CIRCLE_ID" in globals() and "QA_WINDOW_PIXELS" in globals(), "Run Section 00"
name="Baego.tif"
spec=spec_for(name)
out=MAPS/(Path(name).stem+"_M2_"+MODEL_MODE+"_probability_u16_FULL.tif")
assert full_ok(out,spec),("Baego complete map not found",out)
circle_raw=gpd.read_file(A["exhaustive_annotations"]["complete_roi_boundaries"])
assert 0<=QA_CIRCLE_ID<len(circle_raw)
assert circle_raw.geometry.iloc[QA_CIRCLE_ID] is not None
circle=gpd.GeoSeries([circle_raw.geometry.iloc[QA_CIRCLE_ID]],crs=circle_raw.crs)
positives=gpd.read_file(A["exhaustive_annotations"]["positive_polygons"])
roi=gpd.read_file(A["roi"])
with rasterio.open(MOSAICS[name]) as src,rasterio.open(out) as ds:
    assert src.crs==ds.crs and src.transform==ds.transform and src.shape==ds.shape
    circle=circle.to_crs(src.crs)
    x,y=circle.geometry.iloc[0].centroid.coords[0]
    rr,cc=src.index(x,y)
    sz=int(QA_WINDOW_PIXELS);assert sz>=256 and sz%256==0
    r=max(0,min(src.height-sz,rr-sz//2))
    c=max(0,min(src.width-sz,cc-sz//2))
    h=min(sz,src.height);w=min(sz,src.width)
    win=Window(c,r,w,h)
    rgb_raw=src.read([1,2,3],window=win,masked=True)
    rgb=np.moveaxis(rgb_raw.filled(0),0,-1)
    valid=valid_rgb(rgb_raw)
    z=ds.read(1,window=win)
    assert np.array_equal(z!=NODATA,valid)
    prob=np.where(valid,z.astype('float32')/SCALE,np.nan)
    tx=src.window_transform(win)
    def burn(gdf):
        local=gdf.to_crs(src.crs).geometry
        bb=box(*rasterio.windows.bounds(win,src.transform))
        local=local[local.intersects(bb)]
        return rasterize(((g,1) for g in local if g is not None and not g.is_empty),
                         out_shape=(h,w),transform=tx,fill=0,dtype='uint8').astype(bool)
    mask=burn(gpd.GeoDataFrame(geometry=circle))&burn(roi)&valid
    truth=burn(positives)&mask
    assert mask.any(),("Selected circle absent from window",QA_CIRCLE_ID)
    pred=(prob>=0.5)&mask
    tp=int((pred&truth).sum());fp=int((pred&mask&~truth).sum())
    fn=int((~pred&truth).sum())
    precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)
    f1=2*tp/max(2*tp+fp+fn,1);iou=tp/max(tp+fp+fn,1)
    observed=100*truth.sum()/mask.sum()
    estimated=100*np.nansum(prob[mask])/mask.sum()
    print(f'Baego circle {QA_CIRCLE_ID} | scored pixels {mask.sum():,} '
          f'| observed {observed:.2f}% | predicted {estimated:.2f}% '
          f'| bias {estimated-observed:+.2f} pp')
    print(f'Pixel precision {precision:.3f} | recall {recall:.3f} '
          f'| F1 {f1:.3f} | IoU {iou:.3f}')
    fig,axs=plt.subplots(1,3,figsize=(15,5),constrained_layout=True)
    axs[0].imshow(rgb);axs[0].set_title('UAV RGB')
    axs[1].imshow(rgb);axs[1].imshow(np.ma.masked_where(~truth,truth),
                                   cmap='spring',alpha=.7,vmin=0,vmax=1)
    axs[1].contour(mask,levels=[.5],colors='white',linewidths=.8)
    axs[1].set_title('Annotated Musanga (pink), circle (white)')
    im=axs[2].imshow(prob,cmap='viridis',vmin=0,vmax=1)
    axs[2].contour(mask,levels=[.5],colors='white',linewidths=.8)
    axs[2].set_title('Final M2 probability')
    fig.colorbar(im,ax=axs[2],label='P(Musanga)',shrink=.75)
    for ax in axs:ax.axis('off')
    plt.show()
print('QA reads final GeoTIFF only; no model run or checkpoint changes.')


## 07 — Multi-site circle-window QA from finished GeoTIFFs (no inference)
Randomly select annotated circles **within each of the four sites**. Score the *valid pixels inside the intersection of each circle with a 1024px window*; render RGB, reference and prediction. This is a visual/technical **in-sample** QA, not independent CV. Change sample count and random seed in Section 00. A circle may extend beyond the display window; the reported scores therefore describe only its displayed intersection.


In [ ]:
# SECTION 07 — MULTI-SITE QUICK QA: READ COMPLETED GeoTIFFs ONLY
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from IPython.display import display
REF07=ROOT/'Outputs/V3_1/01_reference_data_no_MOS34'
links07=pd.read_csv(REF07/'01_circle_spatial_folds.csv')
circ07=gpd.read_file(A['exhaustive_annotations']['complete_roi_boundaries'])
pos07=gpd.read_file(A['exhaustive_annotations']['positive_polygons'])
roi07=gpd.read_file(A['roi'])
rng07=np.random.default_rng(QA_RANDOM_SEED)
rows07=[]
selection07=[]
for site in ['Baego','MOS','Yangambi','Yoko']:
    available=links07[links07.site==site]
    assert not available.empty,site
    chosen=rng07.choice(available.circle_id.astype(int).to_numpy(),
                        size=min(QA_CIRCLES_PER_SITE,len(available)),replace=False)
    selection07.extend([(site,int(cid)) for cid in sorted(chosen)])
print('Selected circles:',selection07)
def rasterize07(geoms,src,win,h,w):
    bounds=box(*rasterio.windows.bounds(win,src.transform))
    sub=geoms.to_crs(src.crs)
    sub=sub[sub.intersects(bounds)]
    return rasterize(((g,1) for g in sub if g is not None and not g.is_empty),
        out_shape=(h,w),transform=src.window_transform(win),dtype='uint8').astype(bool)
for site,cid in selection07:
    rec=links07.loc[links07.circle_id==cid].iloc[0]
    name=str(rec.mosaic)
    target=MAPS/(Path(name).stem+'_M2_'+MODEL_MODE+'_probability_u16_FULL.tif')
    assert target.exists(),('Missing completed mosaic',name,target)
    with rasterio.open(MOSAICS[name]) as src,rasterio.open(target) as dst:
        assert src.crs==dst.crs and src.transform==dst.transform and src.shape==dst.shape
        circle=circ07.iloc[[cid]].geometry.to_crs(src.crs)
        cx,cy=circle.iloc[0].representative_point().coords[0]
        crow,ccol=src.index(cx,cy)
        h=min(QA_WINDOW_PIXELS,src.height);w=min(QA_WINDOW_PIXELS,src.width)
        rr=max(0,min(src.height-h,crow-h//2));cc=max(0,min(src.width-w,ccol-w//2))
        win=Window(cc,rr,w,h)
        raw=src.read([1,2,3],window=win,masked=True)
        rgb=np.moveaxis(raw.filled(0),0,-1)
        valid=valid_rgb(raw)
        z=dst.read(1,window=win)
        assert np.array_equal(z!=NODATA,valid),(site,cid,'valid mask mismatch')
        probability=np.where(valid,z.astype('float32')/SCALE,np.nan)
        inside=rasterize07(circle,src,win,h,w)&rasterize07(roi07.geometry,src,win,h,w)&valid
        truth=rasterize07(pos07.geometry,src,win,h,w)&inside
        if not inside.any():
            print('SKIP: no scored pixels',site,cid);continue
        pred=(probability>=0.5)&inside
        tp=int((pred&truth).sum());fp=int((pred&inside&~truth).sum())
        fn=int((~pred&truth).sum())
        denom=int(inside.sum())
        obs=100*int(truth.sum())/denom
        est=100*float(np.nansum(probability[inside]))/denom
        p=tp/max(tp+fp,1);r=tp/max(tp+fn,1)
        f1=2*tp/max(2*tp+fp+fn,1)
        iou=tp/max(tp+fp+fn,1)
        rows07.append(dict(site=site,circle=cid,mosaic=name,scored_pixels=denom,
            observed_pct=obs,predicted_pct=est,bias_pp=est-obs,
            precision=p,recall=r,F1=f1,IoU=iou))
        fig,axs=plt.subplots(1,3,figsize=(12,3.8),constrained_layout=True)
        axs[0].imshow(rgb);axs[0].set_title('RGB')
        axs[1].imshow(rgb)
        axs[1].imshow(np.ma.masked_where(~truth,truth),cmap='spring',alpha=.75,vmin=0,vmax=1)
        axs[1].contour(inside,levels=[.5],colors='white',linewidths=.7)
        axs[1].set_title('Reference Musanga (pink)')
        im=axs[2].imshow(probability,cmap='viridis',vmin=0,vmax=1)
        axs[2].contour(inside,levels=[.5],colors='white',linewidths=.7)
        axs[2].set_title('Final M2 probability')
        for ax in axs:ax.axis('off')
        fig.suptitle(f'{site} | circle {cid} | bias {est-obs:+.2f} pp | F1 {f1:.3f}')
        plt.show();plt.close(fig)
qa07=pd.DataFrame(rows07)
print('=== FINAL FULL-MAP QA: sampled circle-window intersections (IN-SAMPLE) ===')
display(qa07.round(3))
if not qa07.empty:
    display(qa07.groupby('site')[['bias_pp','F1','IoU']].mean().round(3))
    qa07.to_csv(OUT/'07_multi_site_circle_window_QA.csv',index=False)
    print('Saved:',OUT/'07_multi_site_circle_window_QA.csv')


## 08 — All 30 exhaustive-circle QA on the final full mosaics
**Primary production-map QA.** Evaluate **all saved exhaustive reference windows** for each of the 30 KVO circles, using the *exact same 512px reference support* as NB02B (circle ∩ ROI ∩ valid RGB). Read predictions directly from the final georeferenced probability GeoTIFFs—**no neural-network inference**. Assert that every circle has the same number of evaluated pixels and observed fractional cover as independent CV. Summarize full-circle cover error and pixel precision/recall/F1/IoU. Compare final full-mosaic QA to the warm-start window-tiled QA without treating either as independent validation. Note: NB02B's exhaustive-window sampling, rather than a single arbitrary display crop, defines the comparable reference footprint.


In [ ]:
# SECTION 08 — COMPLETE REFERENCE SUPPORT / 30 CIRCLES (NO GPU)
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import box
from IPython.display import display
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

REF08=ROOT/'Outputs/V3_1/01_reference_data_no_MOS34'
links08=pd.read_csv(REF08/'01_circle_spatial_folds.csv')
wins08=pd.read_csv(REF08/'01_candidate_windows.csv')
wins08=wins08[wins08.kind=='exhaustive'].copy()
cv08=pd.read_csv(BASE/'02B_spatialCV_all30_paired.csv')
cv08=cv08[cv08.model=='M2'].copy()
cir08=gpd.read_file(A['exhaustive_annotations']['complete_roi_boundaries'])
pos08=gpd.read_file(A['exhaustive_annotations']['positive_polygons'])
roi08=gpd.read_file(A['roi'])
assert len(links08)==30 and cv08.circle_id.nunique()==30
assert set(links08.circle_id)==set(cv08.circle_id)
def burn08(geoms,src,win):
    bounds=box(*rasterio.windows.bounds(win,src.transform))
    ss=geoms[geoms.intersects(bounds)]
    return rasterize(((g,1) for g in ss if g is not None and not g.is_empty),
        out_shape=(int(win.height),int(win.width)),
        transform=src.window_transform(win),fill=0,dtype='uint8').astype(bool)

results08=[]
for name, group in links08.groupby('mosaic',sort=False):
    assert name in MOSAICS,('Unknown reference mosaic',name)
    site=str(group.site.iloc[0])
    output=MAPS/(Path(name).stem+'_M2_'+MODEL_MODE+'_probability_u16_FULL.tif')
    assert output.is_file(),('Missing final full map',output)
    with rasterio.open(MOSAICS[name]) as src,rasterio.open(output) as dst:
        assert dst.crs==src.crs and dst.transform==src.transform and dst.shape==src.shape
        # Project geometries once per mosaic; no repeated shapefile reads
        circles_proj=cir08.to_crs(src.crs).geometry
        positives_proj=pos08.to_crs(src.crs).geometry
        roi_proj=roi08.to_crs(src.crs).geometry
        for cid in sorted(group.circle_id.astype(int)):
            cw=wins08[(wins08.circle_id==cid)&(wins08.mosaic==name)]
            assert len(cw)>0,('No NB02B reference windows',site,cid)
            n=tp=fp=fn=0;sy=sp=0.
            geometry=gpd.GeoSeries([circles_proj.iloc[cid]],crs=src.crs)
            for rec in cw.itertuples(index=False):
                win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
                assert (int(win.width),int(win.height))==(512,512)
                raw=src.read([1,2,3],window=win,masked=True)
                valid=valid_rgb(raw)
                mask=burn08(geometry,src,win)&burn08(roi_proj,src,win)&valid
                if not mask.any():continue
                truth=burn08(positives_proj,src,win)[mask]
                vals=dst.read(1,window=win)[mask]
                assert np.all(vals!=NODATA),(site,cid,'NoData inside evaluated mask')
                probs=vals.astype(np.float64)/SCALE
                hard=probs>=0.5
                n+=len(truth);sy+=int(truth.sum());sp+=float(probs.sum())
                tp+=int((hard&truth).sum())
                fp+=int((hard&~truth).sum())
                fn+=int((~hard&truth).sum())
            ref=cv08.loc[cv08.circle_id==cid].iloc[0]
            assert n==int(ref.valid_pixels),('Mismatched reference pixel count',site,cid,n,int(ref.valid_pixels))
            observed=100*sy/n
            assert abs(observed-float(ref.observed_fraction))<0.025,(
                'Mismatched reference observed cover',site,cid,observed,ref.observed_fraction)
            predicted=100*sp/n
            results08.append(dict(site=site,circle_id=cid,mosaic=name,valid_pixels=n,
                observed_pct=observed,predicted_pct=predicted,bias_pp=predicted-observed,
                MAE_pp=abs(predicted-observed),
                precision=tp/max(1,tp+fp),recall=tp/max(1,tp+fn),
                F1=2*tp/max(1,2*tp+fp+fn),IoU=tp/max(1,tp+fp+fn)))
            print(f'{site:9s} circle {cid:02d} | bias {predicted-observed:+7.3f} pp | '
                  f'F1 {results08[-1]["F1"]:.3f} | {n:,} reference pixels',flush=True)

full08=pd.DataFrame(results08).sort_values(['site','circle_id']).reset_index(drop=True)
assert len(full08)==30 and full08.circle_id.nunique()==30
def site_stats08(g):
    error=g.predicted_pct.to_numpy()-g.observed_pct.to_numpy()
    yy=g.observed_pct.to_numpy()
    rmse=np.sqrt(np.mean(error**2))
    return pd.Series(dict(circles=len(g),R2=r2_score(yy,g.predicted_pct),
        MAE_pp=np.mean(np.abs(error)),RMSE_pp=rmse,bias_pp=np.mean(error),
        mean_precision=g.precision.mean(),mean_recall=g.recall.mean(),
        mean_F1=g.F1.mean(),mean_IoU=g.IoU.mean()))
summary08=full08.groupby('site').apply(site_stats08,include_groups=False).reset_index()
print('\n=== FULL-MOSAIC QA — ALL 30 CIRCLES (IN-SAMPLE) ===')
display(summary08.round(3))
print('\n=== PER-CIRCLE RESULTS ===')
display(full08.round(3))
qa_dir08=OUT/'08_full_circle_QA';qa_dir08.mkdir(exist_ok=True,parents=True)
full08.to_csv(qa_dir08/'all30_full_mosaic_circle_metrics.csv',index=False)
summary08.to_csv(qa_dir08/'all30_full_mosaic_site_summary.csv',index=False)

# Compare with the existing NB02B warm-start tiled-window predictions:
prior=pd.concat([
    pd.read_csv(QA_DIR/'19_baego_warmstart_by_circle.csv').query("method=='warm_epoch2'")
      .rename(columns={'circle':'circle_id','predicted_pct':'prior_predicted_pct',
                       'pixel_F1':'prior_F1'})[['circle_id','prior_predicted_pct','prior_F1']],
    pd.read_csv(QA_DIR/'20_other_sites_warmstart_by_circle.csv')
      .rename(columns={'circle':'circle_id','warmstart_predicted_pct':'prior_predicted_pct',
                       'warmstart_F1':'prior_F1'})[['circle_id','prior_predicted_pct','prior_F1']]
],ignore_index=True)
paired08=full08.merge(prior,on='circle_id',validate='one_to_one')
assert len(paired08)==30
paired08['map_minus_NB02B_cover_pp']=paired08.predicted_pct-paired08.prior_predicted_pct
paired08['map_minus_NB02B_F1']=paired08.F1-paired08.prior_F1
print('\n=== SAME MODEL: FULL-MAP GLOBAL TILING vs NB02B WINDOW TILING ===')
display(paired08.groupby('site')[['map_minus_NB02B_cover_pp','map_minus_NB02B_F1']].agg(['mean','max','min']).round(3))
paired08.to_csv(qa_dir08/'full_map_vs_NB02B_warmstart_by_circle.csv',index=False)

fig,axs=plt.subplots(2,2,figsize=(10,9),constrained_layout=True)
for ax,site in zip(axs.flat,['Baego','MOS','Yangambi','Yoko']):
    g=full08[full08.site==site]
    ax.scatter(g.observed_pct,g.predicted_pct,s=50)
    a=max(10,float(max(g.observed_pct.max(),g.predicted_pct.max()))*1.1)
    ax.plot([0,a],[0,a],'--',color='grey')
    ax.set(xlim=(0,a),ylim=(0,a),title=f'{site} (n={len(g)})',
           xlabel='Observed Musanga cover (%)',ylabel='Full-mosaic predicted cover (%)')
    ax.set_aspect('equal',adjustable='box')
fig.suptitle('Final warm-start M2: complete reference-circle QA (in-sample)')
fig.savefig(qa_dir08/'full_mosaic_all30_scatter_by_site.png',dpi=180)
plt.show();plt.close(fig)
print('\nSaved tables and figure:',qa_dir08)
print('No model inference or training. These are IN-SAMPLE QA, not independent CV.')
